# Finite Volume Method

The finite volume method discretizes the governing conservative transport equations over non-overlapping control volumes $V$ with boundary surface $S$ and outward normal unit vector $\mathbf{n}$.

General integral conservation equation for a scalar transport property $\phi$ (such as velocity components $u, v, w$ or temperature $T$):

$$\frac{\partial}{\partial t} \int_V \rho \phi \, dV + \oint_S \rho \phi (\mathbf{u} \cdot \mathbf{n}) \, dS = \oint_S \Gamma_\phi (\nabla \phi \cdot \mathbf{n}) \, dS + \int_V S_\phi \, dV$$

The convective term $\oint_S \rho \phi (\mathbf{u} \cdot \mathbf{n}) \, dS$ and diffusive term $\oint_S \Gamma_\phi (\nabla \phi \cdot \mathbf{n}) \, dS$ are evaluated across cell faces using interpolation schemes (e.g., First-Order Upwind, Second-Order Upwind, etc.). Flux approximations transform continuous PDEs into algebraic system matrices $[A]\{\phi\} = \{b\}$, solved iteratively using algorithms (e.g, SIMPLE or PISO for pressure-velocity coupling).

Grid quality directly determines numerical diffusion, convergence rate, and physical solution accuracy in room-scale airflows.

| Quality Metric | Definition & Target Range | Data Center Impact |
| --- | --- | --- |
| **Equiangle Skewness** | $S_{e} = \max \left[ \frac{\theta_{\max} - \theta_e}{180 - \theta_e}, \frac{\theta_e - \theta_{\min}}{\theta_e} \right] < 0.85$ | High skewness near perforated tiles causes severe artificial diffusion and local flux errors. |
| **Face Orthogonality** | Angle between cell-center vector and face normal vector ($> 0.15$ or $> 20^\circ$). | Low orthogonality impairs non-orthogonal flux corrections in narrow server aisle gaps. |
| **Aspect Ratio** | Ratio of maximum to minimum cell dimensions ($< 10\text{–}20$ in flow areas). | Keeps cell aspect ratios near $1:1$ in high-mixing shear layers (rack inlets/exhausts). |
| **Wall Boundary Layer ($y^+$)** | Non-dimensional distance $y^+ = \frac{y u_\tau}{\nu}$. | For standard wall functions: $30 < y^+ < 300$. For low-Re wall-resolved models: $y^+ \sim 1$. |

* **Mesh Refinement Strategy:** Apply localized cell refinement at high velocity/thermal gradient regions, including server rack face perforated tiles, CRAH supply/return openings, and thermal plumes above hot aisles.

Convergence requires tracking multiple criteria simultaneously rather than relying solely on residual decay. Unscaled cell errors must drop significantly relative to initial iterations:
* Mass/Continuity: $\text{Residual} < 10^{-3}$
* Momentum ($u, v, w$): $\text{Residual} < 10^{-3} \text{ to } 10^{-4}$
* Energy ($T$): $\text{Residual} < 10^{-6}$

Overall mass flow rate and thermal energy balance over the whole enclosure must reach tight equilibrium:

$$\left\vert{} \frac{\sum \dot{m}_{\text{in}} - \sum \dot{m}_{\text{out}}}{\sum \dot{m}_{\text{in}}} \right\vert{} < 0.1\% \quad \text{and} \quad \left\vert{} \frac{\dot{Q}_{\text{gains}} - \dot{Q}_{\text{cooling}}}{\dot{Q}_{\text{gains}}} \right\vert{} < 1\%$$

Plot local field values over solver iterations at critical locations (e.g., average temperature at a problematic rack inlet or maximum velocity at a perforated tile) until values reach a flat steady-state plateau over $>100$ consecutive iterations.